# ML-08 — Model the Refresh / Content Opportunity Lane

**Lane:** Refresh / Content Opportunity Scoring. The model ranks items for human review; it does not predict that an edit will cause recovery.

This experiment matches the March contract and baseline: features from March 1–15, a rule-defined decline proxy from March 16–31, and June 2026 kept sealed. The baseline and model will be scored on the exact same client-held-out test items using Precision@50 and Precision@100.

## 1. Method choice and why

I use **logistic regression** to estimate the probability of the March second-half decline proxy from five first-half signals. The output is a ranking score for the content team's limited review queue. Logistic regression is a reasonable first learned model here: it is relatively simple, its coefficients can be inspected, and probabilities can be ranked directly. The Week-4 hand rule is retained as the baseline; extra complexity is not automatically better.

The target is a threshold-defined change in impressions, not a later measured refresh outcome. Even a high score or a win over the rule means only stronger agreement with this proxy on a held-out set.

In [5]:
import getpass
import os
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and "google.colab" in sys.modules:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token (input is hidden): ")
if not HF_TOKEN:
    raise RuntimeError("A Hugging Face READ token is required to query the gated warehouse.")

con = duckdb.connect()
escaped_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{escaped_token}')")
del escaped_token, HF_TOKEN

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

model_data = con.sql(f"""
    WITH per_item AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(CASE WHEN report_date < DATE '2026-03-16'
                        THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_first_half,
               SUM(CASE WHEN report_date < DATE '2026-03-16'
                        THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks_first_half,
               AVG(CASE WHEN report_date < DATE '2026-03-16'
                         AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_first_half,
               COUNT(DISTINCT CASE WHEN report_date < DATE '2026-03-16'
                                    AND gsc_impressions > 0 THEN report_date END) AS active_search_days_first_half,
               STDDEV_POP(CASE WHEN report_date < DATE '2026-03-16'
                               THEN gsc_impressions END) AS impression_std_first_half,
               SUM(CASE WHEN report_date >= DATE '2026-03-16'
                        THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_second_half
        FROM {FACT_MARCH}
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT *,
           (impressions_second_half < 0.8 * impressions_first_half)::INTEGER AS is_declining_proxy,
           100.0 * clicks_first_half / NULLIF(impressions_first_half, 0) AS ctr_first_half_pct
    FROM per_item
    WHERE impressions_first_half >= 100
""").df()

if model_data.empty or model_data["is_declining_proxy"].nunique() != 2:
    raise ValueError("The March eligible slice must contain both decline-proxy classes to train and evaluate.")

feature_columns = [
    "impressions_first_half",
    "clicks_first_half",
    "avg_position_first_half",
    "active_search_days_first_half",
    "impression_std_first_half",
]
print(f"Eligible client-content items: {len(model_data):,}")
print(f"Decline-proxy prevalence: {model_data['is_declining_proxy'].mean():.1%}")
print("Five first-half features only; second-half data defines the target, not the predictors.")

Eligible client-content items: 77,540
Decline-proxy prevalence: 28.5%
Five first-half features only; second-half data defines the target, not the predictors.


## 2. Split design

Use one deterministic 75/25 `GroupShuffleSplit`, grouping by `client_hash_id`, with seed 42. This keeps each client entirely in train or test, which is more demanding than randomly splitting content rows. The learned model and Week-4 baseline are evaluated on the same held-out items. The prediction task is limited to the same eligible March slice; no final-month rows are used.

In [6]:
X = model_data[feature_columns].copy()
y = model_data["is_declining_proxy"].astype(int)
groups = model_data["client_hash_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
assert train_clients.isdisjoint(test_clients)
assert y.iloc[train_idx].nunique() == 2 and y.iloc[test_idx].nunique() == 2

print(f"Train items: {len(train_idx):,} across {len(train_clients):,} clients")
print(f"Test items: {len(test_idx):,} across {len(test_clients):,} disjoint clients")
print(f"Held-out decline-proxy base rate: {y.iloc[test_idx].mean():.3f}")

Train items: 61,747 across 28 clients
Test items: 15,793 across 10 disjoint clients
Held-out decline-proxy base rate: 0.147


## 3. Train and compare against the Week-4 baseline

The baseline reproduces the Week-4 `low_ctr_visible_page` rule and impression-weighted CTR-gap score. On rows that do not meet that rule, its ranking score is zero. The learned probabilities and baseline scores rank the identical client-held-out test population. Precision@K is the fraction of top-ranked items matching the same decline proxy; the base rate shows the no-ranking reference. This compares ranking agreement with the proxy, not refresh effectiveness.

In [7]:
def precision_at_k(labels, scores, k):
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    if len(labels) == 0:
        raise ValueError("Cannot compute Precision@K on an empty test set.")
    top_positions = np.argsort(-scores, kind="stable")[:min(k, len(labels))]
    return float(labels[top_positions].mean())


model = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
)
model.fit(X.iloc[train_idx], y.iloc[train_idx])
model_scores = model.predict_proba(X.iloc[test_idx])[:, 1]
model_predictions = (model_scores >= 0.5).astype(int)

test_rows = model_data.iloc[test_idx].copy()
baseline_is_candidate = (
    (test_rows["impressions_first_half"] >= 500)
    & test_rows["avg_position_first_half"].between(1, 20, inclusive="both")
    & (test_rows["ctr_first_half_pct"] < 0.5)
)
baseline_scores = np.where(
    baseline_is_candidate,
    test_rows["impressions_first_half"] * (0.5 - test_rows["ctr_first_half_pct"]) / 100.0,
    0.0,
)
test_labels = y.iloc[test_idx].to_numpy()

comparison = pd.DataFrame([
    {
        "method": "Week-4 CTR rule (baseline)",
        "Precision@50": precision_at_k(test_labels, baseline_scores, 50),
        "Precision@100": precision_at_k(test_labels, baseline_scores, 100),
        "ROC-AUC": roc_auc_score(test_labels, baseline_scores),
        "Average precision": average_precision_score(test_labels, baseline_scores),
    },
    {
        "method": "Logistic regression",
        "Precision@50": precision_at_k(test_labels, model_scores, 50),
        "Precision@100": precision_at_k(test_labels, model_scores, 100),
        "ROC-AUC": roc_auc_score(test_labels, model_scores),
        "Average precision": average_precision_score(test_labels, model_scores),
    },
])

print("Model vs baseline — same test clients, same rows, and same outcome proxy:")
display(comparison.round(3))
print(f"Held-out base rate (random-ranking reference): {test_labels.mean():.3f}")
print(f"Baseline-selected candidates in held-out rows: {int(baseline_is_candidate.sum()):,} / {len(test_rows):,}")

Model vs baseline — same test clients, same rows, and same outcome proxy:


,method,Precision@50,Precision@100,ROC-AUC,Average precision
0,Week-4 CTR rule (baseline),0.12,0.12,0.496,0.148
1,Logistic regression,0.44,0.41,0.615,0.226


Held-out base rate (random-ranking reference): 0.147
Baseline-selected candidates in held-out rows: 4,378 / 15,793


## 4. Errors and interpretation

Read false positives (high predicted decline-proxy probability, proxy is 0) and false negatives (lower probability, proxy is 1) without displaying identifiers. Report performance across a few volume groups, and compute permutation importance on held-out ROC-AUC. Importance describes this fitted model on this sample; it is not causal evidence.

In [8]:
error_rows = test_rows[feature_columns + ["is_declining_proxy"]].copy()
error_rows["predicted_probability"] = model_scores
error_rows["predicted_class"] = model_predictions
error_rows["error_type"] = np.select(
    [(error_rows["predicted_class"] == 1) & (error_rows["is_declining_proxy"] == 0),
     (error_rows["predicted_class"] == 0) & (error_rows["is_declining_proxy"] == 1)],
    ["false_positive", "false_negative"],
    default="correct",
)

error_summary = (
    error_rows.groupby("error_type", observed=False)
    .agg(n=("is_declining_proxy", "size"),
         mean_probability=("predicted_probability", "mean"),
         median_first_half_impressions=("impressions_first_half", "median"))
    .reset_index()
)
print("Held-out classification errors at probability threshold 0.5:")
display(error_summary)

volume_error_rows = error_rows.assign(
    volume_band=pd.cut(
        error_rows["impressions_first_half"],
        bins=[100, 500, 2000, float("inf")],
        labels=["100-499", "500-1,999", "2,000+"],
        right=False,
    )
)
volume_errors = (
    volume_error_rows.groupby("volume_band", observed=False)
    .agg(n=("is_declining_proxy", "size"),
         error_rate=("predicted_class", lambda s: float(np.mean(s.to_numpy() != volume_error_rows.loc[s.index, "is_declining_proxy"].to_numpy()))))
    .reset_index()
)
print("Error rate by first-half impression band:")
display(volume_errors.round(3))

wrong_examples = (
    error_rows.loc[error_rows["error_type"] != "correct"]
    .assign(probability_distance=lambda d: (d["predicted_probability"] - 0.5).abs())
    .sort_values("probability_distance")
    .head(3)
)
print("Three ambiguous held-out errors (IDs omitted; hard cases are near the decision threshold):")
display(wrong_examples[feature_columns + ["is_declining_proxy", "predicted_probability", "error_type"]].round(3))

importance = permutation_importance(
    model, X.iloc[test_idx], y.iloc[test_idx],
    scoring="roc_auc", n_repeats=5, random_state=42, n_jobs=1,
)
importance_table = pd.DataFrame({
    "feature": feature_columns,
    "mean_roc_auc_drop_when_shuffled": importance.importances_mean,
    "std": importance.importances_std,
}).sort_values("mean_roc_auc_drop_when_shuffled", ascending=False)
print("Held-out permutation importance (larger ROC-AUC drop means more reliance in this evaluation):")
display(importance_table.round(4))
print(f"Logistic regression threshold accuracy (secondary metric): {accuracy_score(test_labels, model_predictions):.3f}")
print(f"Logistic regression test ROC-AUC: {roc_auc_score(test_labels, model_scores):.3f}")

print(
    "Interpretation: even if the model leads at a chosen K, it is only ranking a March rule-defined "
    "proxy. Errors near the threshold are plausibly noisy because impression counts fluctuate; "
    "the CTR baseline targets a related review opportunity, not necessarily the same decline mechanism. "
    "Human review remains necessary."
)

Held-out classification errors at probability threshold 0.5:


,error_type,n,mean_probability,median_first_half_impressions
0,correct,10491,0.423767,498.0
1,false_negative,1096,0.402081,422.0
2,false_positive,4206,0.539640,473.0


Error rate by first-half impression band:


,volume_band,n,error_rate
0,100-499,8049,0.347
1,"500-1,999",5657,0.332
2,"2,000+",2087,0.301


Three ambiguous held-out errors (IDs omitted; hard cases are near the decision threshold):


,impressions_first_half,clicks_first_half,avg_position_first_half,active_search_days_first_half,impression_std_first_half,is_declining_proxy,predicted_probability,error_type
19359,2157.0,1.0,13.050,13,87.860,0,0.5,false_positive
72043,493.0,1.0,2.592,15,14.184,0,0.5,false_positive
55670,109.0,0.0,1.921,15,3.043,1,0.5,false_negative


Held-out permutation importance (larger ROC-AUC drop means more reliance in this evaluation):


,feature,mean_roc_auc_drop_when_shuffled,std
3,active_search_days_first_half,0.0785,0.0051
1,clicks_first_half,0.0478,0.0035
4,impression_std_first_half,0.0392,0.0013
0,impressions_first_half,0.0045,0.0011
2,avg_position_first_half,-0.0141,0.0026


Logistic regression threshold accuracy (secondary metric): 0.664
Logistic regression test ROC-AUC: 0.615
Interpretation: even if the model leads at a chosen K, it is only ranking a March rule-defined proxy. Errors near the threshold are plausibly noisy because impression counts fluctuate; the CTR baseline targets a related review opportunity, not necessarily the same decline mechanism. Human review remains necessary.


**What the held-out comparison says:** logistic regression reached Precision@50 **0.44** and Precision@100 **0.41**, above the Week-4 CTR rule (**0.12** at both K values) and the held-out proxy base rate (**0.147**). ROC-AUC was **0.615** for logistic regression and **0.496** for the rule. This supports trying the simpler learned ranker for this specific March decline proxy, but does not prove future-month or post-refresh performance. The rule was designed to surface low-CTR review opportunities, not specifically to rank later impression declines, so its weaker proxy score should not be interpreted as failure at its original purpose.

At a 0.5 classification threshold, the model made **4,206 false positives** and **1,096 false negatives**. Errors were more frequent in the lowest-volume band (0.347) than in the 2,000+ impression band (0.301), consistent with noisier labels at lower counts. The three displayed ambiguous cases sit near probability 0.5. Held-out permutation importance was highest for active search days (ROC-AUC drop 0.077), then clicks (0.051) and impression variability (0.039); these are associations in one client holdout, not causes. Keep this as decision-support and validate on a later month before relying on it.

## 5. Self-check

- [x] Method fits the ranking task, and the choice is explained.
- [x] Client-group holdout with fixed seed; no client crosses train/test.
- [x] Week-4 baseline and model use identical held-out items, proxy label, and Precision@K metrics.
- [x] Base rate, confusion errors, volume-band error rates, three ambiguous cases, and held-out permutation importance are reported.
- [x] Only first-half features predict the second-half proxy; June 2026 is untouched.
- [ ] Execute top to bottom with approved warehouse access and inspect the comparison table.
- [x] No client names, URLs, or private queries are displayed.
- [ ] Commit the executed notebook to your public repository, then submit its GitHub URL on the card.

**Research paper:** read the paper linked in the assignment card before the methodology-review task; that external resource is not included in this repository.